In [2]:
%cd ..

/scratch/big/home/daawic/BSc-Thesis


In [3]:
import os
import torch
import matplotlib.pyplot as plt
from torchvision.transforms import transforms
from project.models import EDMCallum
from project.util.device import get_available_acc
from project.util.plotting import plot_sample
from project.util.data import ReplayMemoryData
from project.util.transforms import Difference
from project.util.metrics import PSNR, MSE

In [4]:
PATH = os.path.join("..", "checkpoints", "diff", "Breakout30.pt")
DATA = os.path.join("..", "checkpoints", "memory", "Breakout.pt")

In [5]:
device = "cuda:1"

In [6]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Pad(2),
    Difference(),
    transforms.Normalize(0.5, 0.5),
])

In [7]:
data = ReplayMemoryData(
    memory=DATA,
    transform=transform,
    cap=500_000,
    train=False,
)

In [32]:
model = EDMCallum.from_checkpoint(PATH, device, U=3).to(device)

In [33]:
n = 16

x = torch.zeros((n, 7, 88, 88), device=device)

for i, img in enumerate(torch.randperm(500_000)[:n]):
    x[i] = data[img].to(device)

mask = torch.ones_like(x, device=device)
mask[:, :, :9] = 0
x_masked = x * mask
x_inpainted = model.inpaint(x, mask)

100%|██████████| 32/32 [00:18<00:00,  1.70it/s]


In [ ]:
psnrs = torch.zeros(n)
mses = torch.zeros(n)

for i in range(n):
    psnrs[i] = PSNR(x[i, :4, :9], x_inpainted[i, :4, :9], torch.tensor(2))
    mses[i] = MSE(x[i, :4, :9], x_inpainted[i, :4, :9])

tensor([2.2482e-02, 3.0391e-02, 4.8878e-02, 1.1842e-02, 9.5293e-06, 5.0761e-03,
        3.6776e-02, 2.2239e-02, 6.3783e-03, 2.0677e-02, 2.6157e-02, 3.8498e-02,
        1.0298e-05, 2.3557e-02, 5.1361e-03, 3.2907e-02])


In [38]:
print(f"MSE: {mses.mean()} (error: {mses.std() / n ** 0.5})")
print(f"MSE: {psnrs.mean()} (error: {psnrs.std() / n ** 0.5})")

MSE: 0.02068835310637951 (error: 0.003697785548865795)
MSE: 27.31409454345703 (error: 2.9076528549194336)
